<a href="https://colab.research.google.com/github/Fdiannz/BT4222-Project-Group-4--MIND/blob/main/step4_advanced_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# step4_advanced_model: NRMS with category and entity features

**Owner:** P5

**Purpose:** a content-based neural recommender built on NRMS (Neural News Recommendation with Multi-Head Self-Attention). It scores each article from its **title words, category and named entities**, matched against the user's **click history**. Because every article is judged by content available at publication time, not by its click count, newly published (cold) articles can be ranked from the moment they appear.

**Inputs:** `data/processed/` from step 1: `news_metadata`, `fit_*` (9–13 Nov), `tune_*` (14 Nov), `eval_*` (15 Nov); `entity_embedding.vec` from the raw MIND folders.

**Outputs (shared):**
- `data/scores/step4_nrms_{tune,eval,train_sample}.parquet`: one row per displayed candidate, columns `impression_id, news_id, score` (raw dot product, higher = more relevant). Used by step 5 (MMR) and step 6 (diagnostics).
- `data/embeddings/step4_nrms_news_vectors.npy` + `step4_nrms_news_ids.parquet`: the model's 256-d article vectors and their article IDs (same row order).
- `outputs/step4_nrms/`: checkpoints, training logs, result tables.

**Runtime:** Colab **GPU** (*Runtime → Change runtime type → T4 GPU*). Run top to bottom and keep the outputs. With `RESUME = True`, finished runs are loaded from their checkpoints instead of being retrained.

### Architecture
NRMS (Wu et al., 2019) represents an article by its title only. Our project needs new articles to compete on content (interim report, Sections 1.2–1.3), and MIND provides two more content signals for every article as soon as it is published, so the news encoder is extended to use all three:

1. **News encoder**
   - *Title:* title words → word vectors → multi-head self-attention between the words → additive attention pools the words into a title vector (standard NRMS).
   - *Category:* learned embeddings of the article's category and of its category_subcategory pair (e.g. *sports_football_nfl*).
   - *Entities (entity encoder):* the Wikidata entities linked in the title and abstract (e.g. *Los Angeles Lakers*). Each entity is described by its pre-trained TransE knowledge-graph vector and its entity-linking features (type, confidence, title or abstract); self-attention between the entities and additive attention pool them into an entity vector, just as the title encoder pools words.
   - The three parts are concatenated and mapped to one 256-d article vector by a linear layer.
2. **User encoder:** each clicked history article goes through the same news encoder → multi-head self-attention between the history articles → additive attention pools them into one user vector (standard NRMS).
3. **Click score** = user vector · candidate article vector.

The contribution of each added input is measured in the ablations (section 12) by removing it, and the full-scale model is compared with standard title-only NRMS (section 11).

### Design rationale

| Component | Why it is used |
|---|---|
| Pre-trained word vectors (word2vec-google-news-300), fine-tuned | Titles are short, so each word must carry meaning. Vectors pre-trained on a large news corpus give even rarely seen words a sensible starting point; fine-tuning then adapts them to what readers click. |
| Multi-head self-attention over words | A word's meaning depends on the words around it ("Apple shares" vs "apple pie"). Each head can model a different kind of relationship between words. |
| Additive attention pooling | Not every word or past click is equally informative. Learned weights pick out the informative ones, and the weights can be inspected (section 13). |
| Self-attention over the click history | Past clicks relate to each other (several clicks on one ongoing story), so each history article is read in the context of the others. |
| 1 click + 4 negatives from the same impression, softmax cross-entropy | Every negative was actually shown and skipped, so non-clicks are genuine negatives rather than articles the user never saw. Scoring the click against the other four turns training into a ranking task, which matches how the model is evaluated. |
| Category and subcategory embeddings | Readers have stable topic interests, and the category is known for every article, including ones with very short or vague titles. The category_subcategory pair is used because some subcategory names (*video*, *causes*) appear under several categories. A subcategory needs at least 10 training articles to get its own embedding; rarer ones share an "other" slot within their category, so no embedding is learned from a handful of examples. |
| Knowledge-graph entity vectors (TransE, fixed) | Entities identify the people, teams and places a story is about, even when two titles share no words. The TransE vectors were trained on the Wikidata graph, not on clicks, so a brand-new article about a known entity gets an informative vector immediately. Keeping them fixed means entities that appear only in test-day articles are represented consistently with those seen in training. |
| Entity-linking features (type, confidence, title vs abstract) | The type tells the model what kind of thing an entity is (a person, an organisation, a place), the confidence how much to trust the link, and the title flag whether the story is mainly about it. Each is known at publication time. |
| Entity encoder: self-attention + additive attention over entities | Entities in one article inform each other (a player and his team), and they are not equally important: in "Lakers beat Clippers in Los Angeles" the teams matter more than the city. Attention learns these weights instead of averaging, and the weights can be inspected. Its benefit over a plain average is tested in section 12. Articles with no entity get a zero vector plus a flag. |
| No article or user ID embeddings | An ID embedding is only learned from clicks, so a new article would have nothing to go on. Every article vector here is built from content available at publication time. |
| Full-slate, per-impression metrics; separate tune and test days | Scoring a sampled subset of candidates makes ranking look easier than it is. Choosing settings on 14 Nov and testing once on 15 Nov keeps model selection away from the test data. |

### Standard NRMS setup vs this notebook

| Setting | Standard NRMS (paper / MIND benchmark) | Here | Why |
|---|---|---|---|
| Input text | Title words, up to 30 | Same | |
| Word vectors | 300-d GloVe, fine-tuned | 300-d **word2vec-google-news-300**, fine-tuned | Trained on Google News, the same domain as MIND |
| News self-attention | 16 heads × 16 dims = 256 | 16 heads, 256 dims after a linear 300→256 projection | PyTorch `MultiheadAttention` keeps its input size, so the projection gives the same 256-d output |
| Additive attention | Query size 200 | Same | |
| Dropout | 0.2 | Same | |
| History length | 50 most recent clicks | Same | |
| Click predictor | Dot product | Same | |
| Training | 1 click + K=4 negatives from the same impression, softmax cross-entropy, Adam | Same (lr 1e-4, batch 64) | |
| Users with no history | Not specified (padding) | Learned "empty user" vector | About 2–3% of impressions |
| Article inputs | Title only | Title + category/subcategory + entity encoder, fused by one linear layer | Content signals available at publication time, for cold articles; each is tested by removal in section 12 |
| Training data | Full training log | Random subset of 1.5M clicks from 9–13 Nov | Colab GPU time; stated in the report |
| Evaluation | AUC, MRR, nDCG@5/10 per impression | Same, via shared `utils/metrics.py`, on full slates | |

### Inputs left out on purpose
- **Hour of day:** it is the same for every candidate in an impression. In a dot-product model it would shift every candidate's score by the same amount, so the ranking, and therefore AUC, MRR and nDCG, could not change. It is only useful in a model where it can interact with article features.
- **Popularity (prior clicks or click rate):** it favours articles that have already been shown many times, which works against the goal of giving new articles a fair chance. Its effect is tested in the feature-based model instead.

## 0. Setup

In [2]:
# --- Setup: run this first, every session ---
from google.colab import drive
drive.mount('/content/drive')

# Fresh copy of the team repo (shared config.py and utils/) on every run
!rm -rf /content/BT4222-Project-Group-4--MIND && git clone -q https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git /content/BT4222-Project-Group-4--MIND
import sys; sys.path.insert(0, '/content/BT4222-Project-Group-4--MIND')
!pip -q install gensim

import config
print("Data folder:", config.PROCESSED_DIR)

Mounted at /content/drive
Data folder: /content/drive/MyDrive/BT4222_Project/data/processed


## 1. Settings

All experiment settings live in `CFG`. `COLS` maps the standard names used in this notebook to the column names in step 1's files.

`RUN` names the checkpoint files. With `RESUME = True`, a run that already finished is loaded from Drive instead of being retrained. **Change `RUN` whenever you change `CFG`**, so an old checkpoint is never resumed with new settings.

In [3]:
import os, re, json, time, math, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from utils.metrics import evaluate, evaluate_by_segment, label_impression_segment

# ---- folders (all from the shared config.py) ----
PROC = config.PROCESSED_DIR
DATA_DIR = getattr(config, "DATA_DIR", os.path.dirname(PROC))
OUTPUT_DIR = getattr(config, "OUTPUT_DIR", os.path.join(os.path.dirname(DATA_DIR), "outputs"))
SCORE_DIR = getattr(config, "SCORES_DIR", os.path.join(DATA_DIR, "scores"))   # shared score files
EMB_DIR = os.path.join(DATA_DIR, "embeddings")                                  # shared article vectors
RUN_DIR = os.path.join(OUTPUT_DIR, "step4_nrms")                                # checkpoints, logs, tables
for d in (SCORE_DIR, EMB_DIR, RUN_DIR):
    os.makedirs(d, exist_ok=True)

FILES = {
    "news": "news_metadata.parquet",
    "fit_imp": "fit_impressions.parquet",   "fit_cand": "fit_candidates_sampled.parquet",
    "tune_imp": "tune_impressions.parquet", "tune_cand": "tune_candidates.parquet",
    "eval_imp": "eval_impressions.parquet", "eval_cand": "eval_candidates.parquet",
}

# standard name -> column name in step 1's files
COLS = {
    "impression_id": "impression_id",
    "news_id": "news_id",
    "label": "label",
    "is_cold": "cold_article",                  # 1 if the article had < config.COLD_THRESHOLD exposures on earlier days
    "prior_count": "prior_exposure_count",      # exposures on earlier days (used only for checks)
    "time": "time",
    "history": "history",
    "impressions": "impressions",               # full slate string 'N1-0 N2-1 ...' in *_impressions
    "title": "title",
    "abstract": "abstract",
    "category": "category",
    "subcategory": "subcategory",
    "title_entities": "title_entities",         # JSON list of linked Wikidata entities
    "abstract_entities": "abstract_entities",
}

CFG = dict(
    text_fields=("title",), field_lens=(30,),  # words kept per text field
    max_hist=50,              # most recent history clicks per user
    neg_k=4,                  # negatives per click, all from the same impression
    d_model=256, n_heads=16, query_dim=200, dropout=0.2,
    lr=1e-4, batch_size=64, epochs=6,
    n_train_clicks=1_500_000, # random subset of fit clicks (compute budget); stated in the report
    n_tune_imps=20_000,       # fixed tune subset scored after each epoch to pick the best epoch
    n_train_slate_imps=20_000,# training impressions re-scored on full slates (train vs test comparison)
    w2v_limit=500_000,        # most frequent word2vec words loaded (saves RAM)
    seed=getattr(config, "SEED", 42),
    # ---- article inputs besides the title (both off = standard title-only NRMS) ----
    use_category=True,        # category + category_subcategory embeddings
    use_entities=True,        # entity encoder over the article's linked entities
    cat_dim=100,              # size of each category embedding
    max_entities=10,          # entities per article (title first); covers 99.8% of articles
    ent_dim=100,              # entity encoder width
    ent_heads=4,              # self-attention heads between an article's entities (100 / 4 = 25 dims each)
    ent_type_dim=20,          # entity type embedding size
    entity_pool="attention",  # "attention" = entity encoder; "mean" = confidence-weighted average (ablation)
    min_subcat_articles=10,   # fewer training articles -> subcategory shares its category's "other" slot
)
RUN = "nrms_v3"             # checkpoint name of the main model (1.5M clicks, 6 epochs)
BASELINE_RUN = "nrms_main_v2"   # standard title-only NRMS, same data and training; loaded for comparison if its checkpoint exists
USE_W2V = True              # False = random word vectors (quick dry run only)
RESUME = True               # load a finished run / continue an interrupted one
RUN_ABLATIONS = True        # section 12; finished ablations reload from their checkpoints

# MIND's pre-trained TransE entity vectors (one file per official split; all come from the same knowledge graph)
RAW_DIR = getattr(config, "RAW_DIR", os.path.join(DATA_DIR, "raw"))
ENTITY_FILES = [os.path.join(getattr(config, k, os.path.join(RAW_DIR, d)), "entity_embedding.vec")
                for k, d in [("TRAIN_DIR", "MINDlarge_train"), ("DEV_DIR", "MINDlarge_dev"), ("TEST_DIR", "MINDlarge_test")]]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"     # mixed precision on GPU: faster, same results up to rounding

def seed_everything(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])
print("Device:", DEVICE, "| run:", RUN)

Device: cuda | run: nrms_v3


## 2. Load step 1 data and run bias checks

Only the needed columns are loaded, and they are renamed to the standard names in `COLS`.

The checks below guard against the most common sources of bias before any training:
1. **No time overlap:** every fit impression is before every tune impression, which is before every test impression.
2. **Cold labels use earlier days only:** within one day, an article's prior exposure count must not change. A count that grows during the day would mean exposures from the same day were counted, so new articles would turn "warm" within minutes.
3. **Candidates have article text:** every scored article exists in `news_metadata`.

In [4]:
def load(name, want):
    # Read only the wanted columns that exist in the file, renamed to standard names.
    path = os.path.join(PROC, FILES[name])
    have = pq.read_schema(path).names
    cols = [COLS[c] for c in want if COLS[c] in have]
    df = pd.read_parquet(path, columns=cols).rename(columns={COLS[c]: c for c in want})
    print(f"{name:10s} {len(df):>12,} rows | columns used: {list(df.columns)}")
    return df, have

NEED = {
    "news": ["news_id", "title"],
    "fit_imp": ["impression_id", "history"], "fit_cand": ["impression_id", "news_id", "label"],
    "tune_imp": ["impression_id", "history"], "tune_cand": ["impression_id", "news_id", "label", "is_cold"],
    "eval_imp": ["impression_id", "history"], "eval_cand": ["impression_id", "news_id", "label", "is_cold"],
}
OPTIONAL = {"news": ["abstract", "category", "subcategory", "title_entities", "abstract_entities"],
            "fit_cand": ["time", "prior_count"],
            "tune_cand": ["category", "time", "prior_count"],
            "eval_cand": ["category", "time", "prior_count"]}

data, schemas = {}, {}
for name in FILES:
    data[name], schemas[name] = load(name, NEED[name] + OPTIONAL.get(name, []))

missing = {n: [c for c in NEED[n] if c not in data[n].columns] for n in NEED}
missing = {n: c for n, c in missing.items() if c}
assert not missing, f"Missing columns {missing}: update COLS in section 1."

news = data["news"].drop_duplicates("news_id").reset_index(drop=True)
fit_cand, fit_imp = data["fit_cand"], data["fit_imp"]
tune_cand, tune_imp = data["tune_cand"], data["tune_imp"]
eval_cand, eval_imp = data["eval_cand"], data["eval_imp"]

# ---- check 1: no time overlap between fit, tune and test ----
if all("time" in d for d in (fit_cand, tune_cand, eval_cand)):
    spans = {n: (d.time.min(), d.time.max()) for n, d in [("fit", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]}
    for n, (a, b) in spans.items():
        print(f"{n:5s} {a} -> {b}")
    assert spans["fit"][1] < spans["tune"][0] and spans["tune"][1] < spans["eval"][0], "splits overlap in time"
    print("check 1 passed: fit < tune < test in time")

# ---- check 2: cold labels are frozen within each day (earlier-day exposures only) ----
for n, d in [("tune", tune_cand), ("eval", eval_cand)]:
    if "prior_count" in d and "time" in d:
        n_values = d.groupby([d["news_id"], d["time"].dt.date])["prior_count"].nunique()
        frozen = (n_values == 1).mean()
        status = "passed" if frozen == 1 else "WARNING: counts change within a day (same-day exposures counted?)"
        print(f"check 2 {n}: {frozen:.1%} of article-days have one prior count -> {status}")

# ---- check 3: every candidate has article text ----
known = set(news["news_id"])
for n, d in [("fit", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]:
    miss = (~d["news_id"].isin(known)).sum()
    assert miss == 0, f"{miss:,} {n} candidates are missing from news_metadata"
print("check 3 passed: all candidates have article text")

for n, d in [("fit (sampled)", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]:
    extra = f", cold share {d.is_cold.mean():.3f}" if "is_cold" in d else ""
    print(f"{n:14s}: {d.impression_id.nunique():>9,} impressions, click rate {d.label.mean():.3f}{extra}")

# ---- keep only the columns used from here on: the full tune and test days have ~30M candidate rows ----
fit_cand = fit_cand[["impression_id", "news_id", "label"]].astype({"label": "int8"})
tune_cand = tune_cand[["impression_id", "news_id", "label", "is_cold"]].astype({"label": "int8", "is_cold": "int8"})
eval_cand = eval_cand[["impression_id", "news_id", "label", "is_cold"]].astype({"label": "int8", "is_cold": "int8"})
del data
gc.collect()

news            104,151 rows | columns used: ['news_id', 'title', 'abstract', 'category', 'subcategory', 'title_entities', 'abstract_entities']
fit_imp       1,801,231 rows | columns used: ['impression_id', 'history']
fit_cand     12,894,210 rows | columns used: ['impression_id', 'news_id', 'label', 'time', 'prior_count']
tune_imp        431,517 rows | columns used: ['impression_id', 'history']
tune_cand    17,400,106 rows | columns used: ['impression_id', 'news_id', 'label', 'is_cold', 'category', 'time', 'prior_count']
eval_imp        376,471 rows | columns used: ['impression_id', 'history']
eval_cand    14,085,557 rows | columns used: ['impression_id', 'news_id', 'label', 'is_cold', 'category', 'time', 'prior_count']
fit   2019-11-09 00:00:00 -> 2019-11-13 23:59:57
tune  2019-11-14 00:00:00 -> 2019-11-14 23:59:59
eval  2019-11-15 00:00:00 -> 2019-11-15 23:59:43
check 1 passed: fit < tune < test in time
check 2 tune: 100.0% of article-days have one prior count -> passed
check 2 eval:

122

## 3. Tokenise article text

Each article becomes a fixed-length array of word IDs (lower-cased, padded with 0), so a batch of titles can be processed as one matrix. Row 0 is a padding article used to fill short histories.

The vocabulary uses article **text only** (titles and abstracts of all articles, no clicks or labels). An article's text is known as soon as it is published, so using test-day titles here is not leakage. Words seen only once map to `<unk>`.

Do not change the tokenizer or vocabulary rules after training: the saved checkpoints depend on the exact vocabulary.

In [5]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
def tokenize(text):
    return TOKEN_RE.findall(str(text).lower()) if isinstance(text, str) else []

# news_id -> row in the token matrix (row 0 = padding article)
NEWS_ROW = pd.Series(np.arange(1, len(news) + 1), index=news["news_id"].to_numpy())

def build_vocab(fields):
    counts = {}
    for f in fields:
        for t in news[f].fillna("") if f in news else []:
            for w in tokenize(t):
                counts[w] = counts.get(w, 0) + 1
    words = sorted(w for w, c in counts.items() if c >= 2)        # rare words -> <unk>
    vocab = {"<pad>": 0, "<unk>": 1}
    vocab.update({w: i + 2 for i, w in enumerate(words)})
    return vocab

def build_tokens(fields, lens, vocab):
    # [n_articles + 1, sum(lens)] word IDs; each field gets its own fixed slot (e.g. 30 title words)
    tok = np.zeros((len(news) + 1, sum(lens)), dtype=np.int64)
    start = 0
    for f, L in zip(fields, lens):
        col = news[f].fillna("") if f in news else pd.Series([""] * len(news))
        for i, text in enumerate(col):
            ids = [vocab.get(w, 1) for w in tokenize(text)][:L]
            tok[i + 1, start:start + len(ids)] = ids
        start += L
    return tok

VOCAB = build_vocab(("title", "abstract") if "abstract" in news else ("title",))
TOKENS = build_tokens(CFG["text_fields"], CFG["field_lens"], VOCAB)
title_len = (TOKENS[1:, :CFG["field_lens"][0]] > 0).sum(1)
print(f"Vocabulary: {len(VOCAB):,} words | articles: {len(news):,} | "
      f"title words kept: median {np.median(title_len):.0f}, empty titles {np.mean(title_len == 0):.2%}")
print("Example:", news.loc[0, "title"], "->", TOKENS[1, :12])

Vocabulary: 53,058 words | articles: 104,151 | title words kept: median 11, empty titles 0.00%
Example: The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By -> [47395  7206 38098 16142 37334  9461  3274 37334 35954 46352  8127     0]


## 3b. Category and entity features

Two more content features per article, each stored as one row per article in the same order as `TOKENS` (row 0 = padding). Neither uses clicks or labels.

**Category IDs.** Each article gets a category ID and a category_subcategory ID. Slots are created only for values that occur in the **fit (training) data**, so every embedding the model has is actually trained:
- a subcategory with fewer than `min_subcat_articles` training articles is mapped to `<category>_other`;
- a category or subcategory that never occurs in training (possible for brand-new sections on the test day) maps to ID 0, a fixed zero vector, rather than to an untrained random vector.

**Entity sequences.** For each article, the entities linked in `title_entities` and `abstract_entities` become a short sequence (title entities first, each in order of first mention, then abstract entities; each entity once; at most `max_entities`). Each entity keeps four inputs for the entity encoder:
- its **TransE vector** (100-d, from MIND's `entity_embedding.vec`): its position in the Wikidata knowledge graph, which places related entities (a team and its players) close together;
- its **type** from entity linking (person, organisation, location, ...), as an ID. Types get a slot only if they occur in the training data, like categories;
- its **linking confidence** (0–1), so the encoder can learn to trust uncertain links less;
- whether it appears in the **title** (1) or only in the abstract (0), since the headline names what the story is mainly about.

Only entities with a TransE vector are kept. The TransE vectors were trained on the knowledge graph, not on clicks, so using the files of all three splits is no more leakage than reading a test-day title. `HAS_ENT` marks articles with at least one usable entity; 99.8% of articles have at most 10.

In [6]:
# ---------- category and category_subcategory IDs (slots only for values seen in training) ----------
def build_category_ids(min_articles):
    cat = news["category"].fillna("").str.lower() if "category" in news else pd.Series([""] * len(news))
    sub = cat + "_" + (news["subcategory"].fillna("").str.lower() if "subcategory" in news else "")
    in_fit = news["news_id"].isin(fit_cand["news_id"]).to_numpy()      # articles in the training data
    n_sub = sub[in_fit].value_counts()
    sub = sub.where(sub.map(n_sub).fillna(0).to_numpy() >= min_articles, cat + "_other")

    def to_ids(labels):
        known = sorted(labels[in_fit].unique())                        # only trained values get a slot
        vocab = {v: i + 1 for i, v in enumerate(known)}                # 0 = unseen -> zero vector
        return np.r_[0, labels.map(vocab).fillna(0).to_numpy(dtype=np.int64)], vocab
    (cat_ids, cat_vocab), (sub_ids, sub_vocab) = to_ids(cat), to_ids(sub)
    return cat_ids, sub_ids, cat_vocab, sub_vocab

CAT_IDS, SUB_IDS, CAT_VOCAB, SUB_VOCAB = build_category_ids(CFG["min_subcat_articles"])
print(f"Categories: {len(CAT_VOCAB)} | category_subcategory slots: {len(SUB_VOCAB)} "
      f"(incl. {sum(k.endswith('_other') for k in SUB_VOCAB)} 'other' slots) | "
      f"articles with an unseen category: {(CAT_IDS[1:] == 0).mean():.2%}")

# ---------- entity sequences: TransE vector + type + confidence + in-title flag ----------
def parse_entities(s):
    # JSON list of entity objects -> [(WikidataId, type, confidence, first offset)]; malformed or empty -> []
    if not isinstance(s, str) or not s.startswith("["):
        return []
    try:
        out = []
        for e in json.loads(s):
            if e.get("WikidataId"):
                ENT_LABEL.setdefault(e["WikidataId"], e.get("Label", e["WikidataId"]))   # for printing only
                out.append((e["WikidataId"], str(e.get("Type", "")), float(e.get("Confidence", 1.0)),
                            min(e.get("OccurrenceOffsets") or [0])))
        return out
    except (ValueError, TypeError, KeyError):
        return []

ENT_LABEL = {}

def article_entities():
    # one list per article: [(qid, type, confidence, in_title)], title entities first, each in order of mention
    cols = [news[c] if c in news else [None] * len(news) for c in ("title_entities", "abstract_entities")]
    out = []
    for t_s, a_s in zip(*cols):
        seen, ents = set(), []
        for s, in_title in ((t_s, 1.0), (a_s, 0.0)):
            for qid, typ, conf, _ in sorted(parse_entities(s), key=lambda e: e[3]):
                if qid not in seen:
                    seen.add(qid)
                    ents.append((qid, typ, conf, in_title))
        out.append(ents)
    return out

def load_transe(files, wanted):
    vecs = {}
    for path in files:
        if not os.path.exists(path):
            print("  not found (skipped):", path)
            continue
        with open(path) as f:
            for line in f:
                parts = line.split()
                if parts and parts[0] in wanted and parts[0] not in vecs:   # first file wins
                    vecs[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
        print(f"  read {path}: {len(vecs):,} needed entities so far")
    return vecs

def build_entity_inputs(article_ents, transe, max_ent):
    # Arrays with one row per article (row 0 = padding), max_ent entity slots each; slot value 0 = empty
    qids = sorted(transe)
    table = np.zeros((len(qids) + 1, len(transe[qids[0]])), dtype=np.float32)     # row 0 = padding entity
    table[1:] = np.stack([transe[q] for q in qids])
    q_row = {q: i + 1 for i, q in enumerate(qids)}
    in_fit = news["news_id"].isin(fit_cand["news_id"]).to_numpy()
    fit_types = sorted({t for ents, f in zip(article_ents, in_fit) if f for _, t, _, _ in ents})
    type_id = {t: i + 1 for i, t in enumerate(fit_types)}                       # 0 = unseen type -> zero vector
    n = len(article_ents) + 1
    idx, typ = np.zeros((n, max_ent), dtype=np.int64), np.zeros((n, max_ent), dtype=np.int64)
    conf, title = np.zeros((n, max_ent), dtype=np.float32), np.zeros((n, max_ent), dtype=np.float32)
    for i, ents in enumerate(article_ents):
        ents = [e for e in ents if e[0] in q_row][:max_ent]
        for j, (q, t, c, tt) in enumerate(ents):
            idx[i + 1, j], typ[i + 1, j], conf[i + 1, j], title[i + 1, j] = q_row[q], type_id.get(t, 0), c, tt
    return {"ent_table": table, "ent_idx": idx, "ent_type": typ, "ent_conf": conf, "ent_title": title,
            "n_types": len(type_id), "qids": qids}, type_id

ARTICLE_ENTS = article_entities()
TRANSE = load_transe(ENTITY_FILES, {e[0] for ents in ARTICLE_ENTS for e in ents})
if TRANSE:
    ENT_INPUTS, ENT_TYPES = build_entity_inputs(ARTICLE_ENTS, TRANSE, CFG["max_entities"])
    HAS_ENT = (ENT_INPUTS["ent_idx"][:, 0] > 0).astype(np.float32)
    n_ent = (ENT_INPUTS["ent_idx"][1:] > 0).sum(1)
    print(f"Entities: {len(TRANSE):,} with a TransE vector ({ENT_INPUTS['ent_table'].shape[1]}-d), "
          f"{len(ENT_TYPES)} entity types | articles with a linked entity: {np.mean([len(e) > 0 for e in ARTICLE_ENTS]):.1%}, "
          f"with a usable entity: {HAS_ENT[1:].mean():.1%} | entities per article: median {np.median(n_ent):.0f}, "
          f"cut at {CFG['max_entities']}: {np.mean([len(e) > CFG['max_entities'] for e in ARTICLE_ENTS]):.2%} of articles")
    # coverage for cold vs warm test candidates: entities matter most where click history is missing
    rows = NEWS_ROW.reindex(eval_cand["news_id"].to_numpy()).to_numpy()
    for flag, name in [(1, "cold"), (0, "warm")]:
        print(f"  test {name} candidates with an entity: {HAS_ENT[rows[eval_cand.is_cold.to_numpy() == flag]].mean():.1%}")
else:
    ENT_INPUTS, HAS_ENT = None, None
    print("No entity_embedding.vec found: the main model (use_entities=True) needs it. "
          "Put the file in data/raw/MINDlarge_train/ (and the dev/test folders), or set use_entities=False.")
del TRANSE, ARTICLE_ENTS
news = news.drop(columns=[c for c in ("title_entities", "abstract_entities", "url") if c in news])
gc.collect()

FEATURES = {"cat_ids": CAT_IDS, "sub_ids": SUB_IDS, "n_cat": len(CAT_VOCAB), "n_sub": len(SUB_VOCAB),
            "entities": ENT_INPUTS, "has_ent": HAS_ENT}

Categories: 16 | category_subcategory slots: 139 (incl. 15 'other' slots) | articles with an unseen category: 0.00%
  read /content/drive/MyDrive/BT4222_Project/data/raw/MINDlarge_train/entity_embedding.vec: 42,007 needed entities so far
  read /content/drive/MyDrive/BT4222_Project/data/raw/MINDlarge_dev/entity_embedding.vec: 42,628 needed entities so far
  read /content/drive/MyDrive/BT4222_Project/data/raw/MINDlarge_test/entity_embedding.vec: 42,628 needed entities so far
Entities: 42,628 with a TransE vector (100-d), 22 entity types | articles with a linked entity: 88.3%, with a usable entity: 87.9% | entities per article: median 2, cut at 10: 0.22% of articles
  test cold candidates with an entity: 83.5%
  test warm candidates with an entity: 78.0%


## 4. Word vectors: word2vec-google-news-300

Each vocabulary word starts from its pre-trained word2vec vector, which was trained on Google News, the same domain as MIND. Words missing from word2vec (including stopwords such as *a* and *of*) start from small random vectors. All vectors are fine-tuned during training.

The download (about 1.6 GB) happens once; the resulting matrix is cached on Drive.

In [7]:
EMB_DIM = 300
def build_word_vectors(vocab):
    cache = os.path.join(RUN_DIR, f"word2vec_init_{len(vocab)}.npy")
    if os.path.exists(cache):
        print("Loaded cached word vectors:", cache)
        return np.load(cache)
    rng = np.random.default_rng(CFG["seed"])
    emb = rng.normal(0, 0.1, (len(vocab), EMB_DIM)).astype(np.float32)
    emb[0] = 0.0                                          # padding word
    if not USE_W2V:
        print("USE_W2V=False: random word vectors (dry run only)")
        return emb
    import gensim.downloader as api
    from gensim.models import KeyedVectors
    path = api.load("word2vec-google-news-300", return_path=True)
    kv = KeyedVectors.load_word2vec_format(path, binary=True, limit=CFG["w2v_limit"])
    hits = 0
    for w, i in vocab.items():
        if i < 2:
            continue
        for form in (w, w.capitalize(), w.upper()):        # word2vec is case-sensitive
            if form in kv.key_to_index:
                emb[i] = kv[form]; hits += 1
                break
    print(f"word2vec coverage: {hits:,}/{len(vocab) - 2:,} words ({hits / (len(vocab) - 2):.1%})")
    np.save(cache, emb)
    del kv
    return emb

WORD_EMB = build_word_vectors(VOCAB)

Loaded cached word vectors: /content/drive/MyDrive/BT4222_Project/outputs/step4_nrms/word2vec_init_53058.npy


## 5. User histories

Each impression's click history becomes up to 50 article rows (most recent kept), padded with 0. In MIND, the history lists clicks made **before** the impression, so it never contains the impression's own outcome.

In [8]:
def parse_history(h):
    if h is None or (isinstance(h, float) and np.isnan(h)):
        return []
    return h.split() if isinstance(h, str) else list(h)

def build_history(imp_df, impression_ids=None):
    # Returns (H, index): H[n_imp, max_hist] holds article rows; index maps impression_id -> row of H.
    df = imp_df if impression_ids is None else imp_df[imp_df.impression_id.isin(impression_ids)]
    df = df.drop_duplicates("impression_id")
    H = np.zeros((len(df), CFG["max_hist"]), dtype=np.int64)
    for i, h in enumerate(df["history"].to_numpy()):
        ids = parse_history(h)[-CFG["max_hist"]:]           # keep the most recent clicks
        rows = NEWS_ROW.reindex(ids).fillna(0).to_numpy(dtype=np.int64) if ids else []
        H[i, :len(rows)] = rows
    return H, pd.Series(np.arange(len(df)), index=df["impression_id"].to_numpy())

tune_H, tune_idx = build_history(tune_imp)
eval_H, eval_idx = build_history(eval_imp)
for name, H in [("tune", tune_H), ("eval", eval_H)]:
    n = (H > 0).sum(1)
    print(f"{name}: {len(H):,} impressions | history length median {np.median(n):.0f}, empty {np.mean(n == 0):.2%}")

tune: 431,517 impressions | history length median 20, empty 2.64%
eval: 376,471 impressions | history length median 19, empty 2.99%


## 6. Training samples: 1 click + 4 negatives from the same impression

Each click in `fit_candidates_sampled` becomes one sample with 4 unclicked articles **from the same impression**, so every negative was actually shown to the user (interim report, Section 3.3). When an impression has fewer than 4 unclicked articles, some are reused; the share is printed.

The click is always placed first (target index 0). This creates no position bias: the model scores each candidate separately and never sees its position.

In [9]:
def build_train_samples(cand, n_clicks, k, seed):
    # Returns (impression_id per sample, C[n, 1 + k] article rows with the click in column 0).
    rng = np.random.default_rng(seed)
    cand = cand.assign(row=NEWS_ROW.reindex(cand["news_id"].to_numpy()).fillna(0).to_numpy(dtype=np.int64))
    pos = cand[cand.label == 1]
    if n_clicks and n_clicks < len(pos):
        pos = pos.sample(n=n_clicks, random_state=seed)                            # random click subset
    neg = cand[(cand.label == 0) & cand.impression_id.isin(pos.impression_id)]
    neg = neg.assign(r=rng.random(len(neg))).sort_values(["impression_id", "r"])   # shuffle within impression
    imp_ids, start, count = np.unique(neg.impression_id.to_numpy(), return_index=True, return_counts=True)
    where = pd.Series(np.arange(len(imp_ids)), index=imp_ids)
    pos = pos[pos.impression_id.isin(imp_ids)]                                     # need at least one negative
    g = where.reindex(pos.impression_id.to_numpy()).to_numpy()
    j = pos.groupby("impression_id").cumcount().to_numpy()                         # j-th click of its impression
    offs = (j[:, None] * k + np.arange(k)[None, :]) % count[g][:, None]            # different negatives per click
    neg_rows = neg["row"].to_numpy()[start[g][:, None] + offs]
    C = np.concatenate([pos["row"].to_numpy()[:, None], neg_rows], axis=1)
    return pos["impression_id"].to_numpy(), C

train_imp_ids, train_C = build_train_samples(fit_cand, CFG["n_train_clicks"], CFG["neg_k"], CFG["seed"])
fit_H, fit_idx = build_history(fit_imp, impression_ids=np.unique(train_imp_ids))
train_H_rows = fit_idx.reindex(train_imp_ids).to_numpy()
assert not np.isnan(train_H_rows.astype(float)).any(), "some training impressions have no history row"
reused = np.mean([len(set(r[1:])) < CFG["neg_k"] for r in train_C[:200_000]])
print(f"Training samples: {len(train_C):,} clicks x {train_C.shape[1]} candidates from "
      f"{len(np.unique(train_imp_ids)):,} impressions | samples with a reused negative: {reused:.1%}")

Training samples: 1,500,000 clicks x 5 candidates from 1,146,261 impressions | samples with a reused negative: 10.0%


## 7. Model

- `AdditiveAttention`: scores each item with a small learned query vector and returns the weighted average and the weights (so attention can be inspected in section 13).
- `NewsEncoder`: word vectors → dropout → 300→256 projection → 16-head self-attention over the words → dropout → additive attention over the words.
- `UserEncoder`: 16-head self-attention over the history article vectors → dropout → additive attention. Users with no history get a learned "empty user" vector.
- `EntityEncoder`: reads an article's entities the way `NewsEncoder` reads its words. Each entity's input is its fixed TransE vector, a learned embedding of its entity type, its linking confidence and its in-title flag; a linear layer projects this to 100 dimensions, 4-head self-attention lets entities inform each other (a player and his team), and additive attention weights the entities that matter most for the article. Articles without entities get a zero vector. With `entity_pool="mean"` (ablation only), the attention layers are replaced by a confidence-weighted average.
- `NRMS.encode_news`: the article vector. The title vector from `NewsEncoder` is concatenated with the two category embeddings, the entity vector from `EntityEncoder` and a `has_entity` flag; a linear layer maps the result to 256 dimensions. Every article, in the history and among the candidates, goes through the same function, so a new candidate is represented exactly like an article the user has clicked.
- `NRMS.forward`: encodes each distinct article in the batch once, then returns **raw** dot-product scores. `F.cross_entropy` applies the softmax itself, which is numerically more stable than taking the softmax and the log separately.

Layer names must stay as they are, because the saved checkpoints refer to them. The category and entity layers exist only when their switch is on, so title-only checkpoints (the baseline and the ablations) load with the same code.

In [10]:
class AdditiveAttention(nn.Module):
    def __init__(self, dim, query_dim):
        super().__init__()
        self.proj = nn.Linear(dim, query_dim)
        self.query = nn.Linear(query_dim, 1, bias=False)

    def forward(self, x, mask):                       # x [N, L, d], mask [N, L] True = real item
        a = self.query(torch.tanh(self.proj(x))).squeeze(-1)
        a = a.masked_fill(~mask, -1e4)                # padding gets ~0 weight; -1e4 is safe in float16
        w = torch.softmax(a, dim=-1)
        return (w.unsqueeze(-1) * x).sum(1), w


def _safe_mask(mask):
    # Attention cannot run over a row with no real items, so unmask the first slot of empty rows.
    empty = ~mask.any(1)
    mask = mask.clone()
    mask[empty, 0] = True
    return mask, empty


class NewsEncoder(nn.Module):
    def __init__(self, word_emb, d_model, n_heads, query_dim, dropout):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.as_tensor(word_emb), freeze=False, padding_idx=0)
        self.proj = nn.Linear(word_emb.shape[1], d_model)
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.att = AdditiveAttention(d_model, query_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, tokens, return_weights=False):  # tokens [N, L] word IDs
        mask, _ = _safe_mask(tokens > 0)
        x = self.proj(self.drop(self.emb(tokens)))
        h, _ = self.mha(x, x, x, key_padding_mask=~mask, need_weights=False)
        v, w = self.att(self.drop(h), mask)
        return (v, w) if return_weights else v


class EntityEncoder(nn.Module):
    # An article's entities, read like the words of its title: each entity = [TransE vector, type embedding,
    # linking confidence, in-title flag] -> projection -> self-attention between the entities -> additive attention.
    def __init__(self, ent_in_dim, n_types, type_dim, d_ent, n_heads, query_dim, dropout, pool="attention"):
        super().__init__()
        self.type_emb = nn.Embedding(n_types + 1, type_dim, padding_idx=0)     # 0 = padding / unseen type
        self.proj = nn.Linear(ent_in_dim + type_dim + 2, d_ent)
        self.pool = pool
        if pool == "attention":
            self.mha = nn.MultiheadAttention(d_ent, n_heads, dropout=dropout, batch_first=True)
            self.att = AdditiveAttention(d_ent, query_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, vecs, types, conf, in_title, return_weights=False):   # [N, E, 100], [N, E] x 3
        real = vecs.abs().sum(-1) > 0                             # padding slots point to the zero row
        x = torch.cat([vecs, self.type_emb(types), conf.unsqueeze(-1), in_title.unsqueeze(-1)], dim=-1)
        x = self.proj(self.drop(x))
        mask, empty = _safe_mask(real)
        if self.pool == "attention":
            h, _ = self.mha(x, x, x, key_padding_mask=~mask, need_weights=False)
            v, w = self.att(self.drop(h), mask)
        else:                                                    # ablation: confidence-weighted average
            w = conf * real
            w = w / w.sum(1, keepdim=True).clamp_min(1e-6)
            v = (w.unsqueeze(-1).to(x.dtype) * x).sum(1)
        v = v * (~empty).unsqueeze(-1).to(v.dtype)              # no entity -> zero vector
        return (v, w) if return_weights else v


class UserEncoder(nn.Module):
    def __init__(self, d_model, n_heads, query_dim, dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.att = AdditiveAttention(d_model, query_dim)
        self.drop = nn.Dropout(dropout)
        self.empty_user = nn.Parameter(torch.zeros(d_model))

    def forward(self, hist_vecs, hist_mask, return_weights=False):   # [B, H, d], [B, H]
        mask, empty = _safe_mask(hist_mask)
        h, _ = self.mha(hist_vecs, hist_vecs, hist_vecs, key_padding_mask=~mask, need_weights=False)
        u, w = self.att(self.drop(h), mask)
        u = torch.where(empty.unsqueeze(1), self.empty_user.to(u.dtype).expand_as(u), u)
        return (u, w) if return_weights else u


class NRMS(nn.Module):
    def __init__(self, word_emb, tokens, cfg, features=None):
        super().__init__()
        self.register_buffer("tokens", torch.as_tensor(tokens), persistent=False)
        self.news_encoder = NewsEncoder(word_emb, cfg["d_model"], cfg["n_heads"], cfg["query_dim"], cfg["dropout"])
        self.user_encoder = UserEncoder(cfg["d_model"], cfg["n_heads"], cfg["query_dim"], cfg["dropout"])

        # ---- category and entity inputs; created only when switched on, so title-only checkpoints still load ----
        self.use_cat = cfg.get("use_category", False)
        self.use_ent = cfg.get("use_entities", False)
        extra = 0
        if self.use_cat:
            self.register_buffer("cat_ids", torch.as_tensor(features["cat_ids"]), persistent=False)
            self.register_buffer("sub_ids", torch.as_tensor(features["sub_ids"]), persistent=False)
            self.cat_emb = nn.Embedding(features["n_cat"] + 1, cfg["cat_dim"], padding_idx=0)   # 0 = unseen
            self.sub_emb = nn.Embedding(features["n_sub"] + 1, cfg["cat_dim"], padding_idx=0)
            extra += 2 * cfg["cat_dim"]
        if self.use_ent:
            if features is None or features["entities"] is None:
                raise ValueError("use_entities=True needs entity_embedding.vec (see section 3b)")
            ent = features["entities"]
            self.register_buffer("ent_table", torch.as_tensor(ent["ent_table"]), persistent=False)  # fixed TransE
            for k in ("ent_idx", "ent_type", "ent_conf", "ent_title"):
                self.register_buffer(k, torch.as_tensor(ent[k]), persistent=False)
            self.register_buffer("has_ent", torch.as_tensor(features["has_ent"]), persistent=False)
            self.entity_encoder = EntityEncoder(ent["ent_table"].shape[1], ent["n_types"], cfg["ent_type_dim"],
                                                cfg["ent_dim"], cfg["ent_heads"], cfg["query_dim"], cfg["dropout"],
                                                pool=cfg.get("entity_pool", "attention"))
            extra += cfg["ent_dim"] + 1                                   # + the has_entity flag
        if extra:
            self.feat_drop = nn.Dropout(cfg["dropout"])
            self.fuse = nn.Linear(cfg["d_model"] + extra, cfg["d_model"])

    def encode_news(self, rows):
        v = self.news_encoder(self.tokens[rows])
        if not (self.use_cat or self.use_ent):
            return v                                                       # standard NRMS
        side, flags = [], []
        if self.use_cat:
            side += [self.cat_emb(self.cat_ids[rows]), self.sub_emb(self.sub_ids[rows])]
        if self.use_ent:
            side.append(self.encode_entities(rows))                              # no entity -> zero vector
            flags.append(self.has_ent[rows].unsqueeze(-1))                       # ... and flag 0
        side = self.feat_drop(torch.cat([s.to(v.dtype) for s in side], dim=-1)) # same dropout as the words
        return self.fuse(torch.cat([v, side] + [f.to(v.dtype) for f in flags], dim=-1))

    def encode_entities(self, rows, return_weights=False):
        return self.entity_encoder(self.ent_table[self.ent_idx[rows]], self.ent_type[rows],
                                   self.ent_conf[rows], self.ent_title[rows], return_weights=return_weights)

    def forward(self, hist_rows, cand_rows):         # [B, H], [B, C] article rows
        B, H = hist_rows.shape
        uniq, inv = torch.unique(torch.cat([hist_rows.reshape(-1), cand_rows.reshape(-1)]), return_inverse=True)
        vecs = self.encode_news(uniq)                  # each distinct article encoded once per batch
        hist_vecs = vecs[inv[:B * H]].view(B, H, -1)
        cand_vecs = vecs[inv[B * H:]].view(B, cand_rows.shape[1], -1)
        user = self.user_encoder(hist_vecs, hist_rows > 0)
        return (cand_vecs * user.unsqueeze(1)).sum(-1)  # raw scores [B, C]

for _name, _cfg in [("Our model (title + category + entities)", CFG),
                    ("Standard NRMS (title only)", dict(CFG, use_category=False, use_entities=False))]:
    _m = NRMS(WORD_EMB, TOKENS, _cfg, FEATURES)
    print(f"{_name}: {sum(p.numel() for p in _m.parameters()):,} parameters "
          f"(of which word vectors {_m.news_encoder.emb.weight.numel():,})")
del _m

Our model (title + category + entities): 16,856,356 parameters (of which word vectors 15,917,400)
Standard NRMS (title only): 16,624,248 parameters (of which word vectors 15,917,400)


## 8. Scoring full slates

To score a split, every article is encoded once, each impression's user vector is computed once, and every displayed candidate is scored by a dot product. All candidates are scored (full slates, no sampling). Score files have the shared format `impression_id, news_id, score`. `with_scores` attaches the scores to the labels with the same checks as `utils/metrics.attach_scores` (one finite score per candidate), but without a merge, because a merge would hold a second copy of the 14M-row test day in memory.

In [11]:
@torch.no_grad()
def encode_all_news(model, batch=2048):
    # [n_articles + 1, d] article vectors; row r belongs to news.loc[r - 1], row 0 is padding
    model.eval()
    out = []
    for s in range(0, model.tokens.shape[0], batch):
        rows = torch.arange(s, min(s + batch, model.tokens.shape[0]), device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            out.append(model.encode_news(rows).float())
    return torch.cat(out)

@torch.no_grad()
def score_candidates(model, cand, H, h_index, batch=1024):
    model.eval()
    news_vecs = encode_all_news(model)
    imp_codes, imp_ids = pd.factorize(cand["impression_id"])
    imp_codes = np.array(imp_codes, dtype=np.int64)
    h_rows = h_index.reindex(imp_ids).to_numpy()
    if np.isnan(h_rows.astype(float)).any():
        raise ValueError("some candidate impressions have no history row")
    h_rows = h_rows.astype(np.int64)
    users = []                                          # one user vector per impression
    for s in range(0, len(imp_ids), batch):
        hist = torch.as_tensor(H[h_rows[s:s + batch]], device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            users.append(model.user_encoder(news_vecs[hist], hist > 0).float())
    users = torch.cat(users)
    news_rows = NEWS_ROW.reindex(cand["news_id"].to_numpy()).fillna(0).to_numpy(dtype=np.int64)
    scores = np.empty(len(cand), dtype=np.float32)
    for s in range(0, len(cand), 200_000):
        u = users[torch.as_tensor(imp_codes[s:s + 200_000], device=DEVICE)]
        v = news_vecs[torch.as_tensor(news_rows[s:s + 200_000], device=DEVICE)]
        scores[s:s + 200_000] = (u * v).sum(-1).cpu().numpy()
    return pd.DataFrame({"impression_id": cand["impression_id"].to_numpy(),
                         "news_id": cand["news_id"].to_numpy(), "score": scores})

def with_scores(cand, s):
    # Attach scores to the labelled candidates. score_candidates returns them in the candidates' own row
    # order, so they are attached directly after checking that order, rather than by a merge
    # (utils/metrics.attach_scores): the same checks without a second copy of a 14M-row table in memory.
    assert len(s) == len(cand), "every candidate needs exactly one score"
    assert (s["impression_id"].to_numpy() == cand["impression_id"].to_numpy()).all(), "row order differs"
    assert (s["news_id"].to_numpy() == cand["news_id"].to_numpy()).all(), "row order differs"
    assert np.isfinite(s["score"].to_numpy()).all(), "NaN or infinite scores"
    out = cand[[c for c in ("impression_id", "label", "is_cold") if c in cand]].copy()
    out["score"] = s["score"].to_numpy()
    return out

def sample_impressions(cand, n, seed):
    ids = cand["impression_id"].drop_duplicates()
    return cand[cand["impression_id"].isin(ids.sample(n=min(n, len(ids)), random_state=seed))]

tune_small = sample_impressions(tune_cand, CFG["n_tune_imps"], CFG["seed"])   # fixed subset for epoch selection
print(f"Tune subset for model selection: {tune_small.impression_id.nunique():,} impressions")

Tune subset for model selection: 20,000 impressions


## 9. Training loop

- **Loss:** softmax cross-entropy over [click, 4 negatives] with target 0, on raw scores.
- **Optimiser:** Adam, learning rate 1e-4, mini-batches of 64.
- **Model selection:** after each epoch the model scores the fixed tune subset (14 Nov, full slates), and the epoch with the best **tune AUC** is kept. Training scores are never used for selection, and the test day is never seen during training or selection.
- **Checkpoints:** saved to Drive after every epoch; `RESUME = True` continues an interrupted run.

In [12]:
def train_nrms(cfg, run_name, train_H, train_H_rows, train_C, tune=None, select_on_tune=True, log_every=500):
    tune_c, tune_h, tune_i = tune if tune is not None else (tune_small, tune_H, tune_idx)
    seed_everything(cfg["seed"])
    model = NRMS(WORD_EMB, TOKENS, cfg, FEATURES).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    last_ckpt = os.path.join(RUN_DIR, f"{run_name}_last.pt")
    best_ckpt = os.path.join(RUN_DIR, f"{run_name}_best.pt")
    log, start_epoch, best_auc = [], 0, -1.0
    if RESUME and os.path.exists(last_ckpt):
        st = torch.load(last_ckpt, map_location=DEVICE)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        log, start_epoch, best_auc = st["log"], st["epoch"] + 1, st["best_auc"]
        print(f"Resumed {run_name} after epoch {start_epoch}")

    H_dev = torch.as_tensor(train_H, device=DEVICE)
    rows_dev = torch.as_tensor(train_H_rows.astype(np.int64), device=DEVICE)
    C_dev = torch.as_tensor(train_C, device=DEVICE)
    target = torch.zeros(cfg["batch_size"], dtype=torch.long, device=DEVICE)   # the click is column 0
    n, bs = len(C_dev), cfg["batch_size"]
    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        t0, total, perm = time.time(), 0.0, torch.randperm(n, device=DEVICE)
        for b, s in enumerate(range(0, n, bs)):
            idx = perm[s:s + bs]
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                logits = model(H_dev[rows_dev[idx]], C_dev[idx])
                loss = F.cross_entropy(logits.float(), target[:len(idx)])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total += loss.item() * len(idx)
            if log_every and b % log_every == 0:
                print(f"  epoch {epoch + 1} batch {b:>6}/{math.ceil(n / bs)} loss {loss.item():.4f}")
        row = {"epoch": epoch + 1, "train_loss": total / n}
        if select_on_tune:
            tune_res = evaluate(with_scores(tune_c, score_candidates(model, tune_c, tune_h, tune_i)), "score")
            row.update({f"tune_{k}": v for k, v in tune_res.items()})
            improved = tune_res["auc"] > best_auc
            best_auc = max(best_auc, tune_res["auc"])
        else:
            improved = True                                       # fixed epochs: keep the latest
        row["minutes"] = (time.time() - t0) / 60
        log.append(row)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
        if improved:
            torch.save({"model": model.state_dict(), "cfg": cfg, "epoch": epoch + 1}, best_ckpt)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch,
                    "log": log, "best_auc": best_auc}, last_ckpt)

    best = torch.load(best_ckpt, map_location=DEVICE)
    model.load_state_dict(best["model"])
    log = pd.DataFrame(log)
    log.to_csv(os.path.join(RUN_DIR, f"{run_name}_log.csv"), index=False)
    model.best_epoch = int(best["epoch"])
    msg = f"(tune AUC {log.loc[log.epoch == model.best_epoch, 'tune_auc'].item():.4f})" if select_on_tune else "(fixed)"
    print(f"{run_name}: using epoch {model.best_epoch} {msg}")
    return model, log

## 10. Train the main model (fit: 9–13 Nov)

Trains the full model (title + category + entities). If `RUN` already finished, this cell only loads the best checkpoint.

In [13]:
model, train_log = train_nrms(CFG, RUN, fit_H, train_H_rows, train_C)
with open(os.path.join(RUN_DIR, f"{RUN}_config.json"), "w") as f:
    json.dump({**{k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
               "best_epoch": model.best_epoch, "trained_on": "fit 9-13 Nov"}, f, indent=2)
train_log

Resumed nrms_v3 after epoch 6
nrms_v3: using epoch 4 (tune AUC 0.6816)


,epoch,train_loss,tune_auc,tune_mrr,tune_ndcg@5,tune_ndcg@10,tune_n_impressions,tune_n_skipped,minutes
0,1,1.327675,0.6655,0.3049,0.3294,0.3909,20000,0,18.648630
1,2,1.279838,0.6735,0.3082,0.3351,0.3974,20000,0,18.423544
2,3,1.261572,0.6764,0.3128,0.3394,0.4021,20000,0,18.470229
3,4,1.249253,0.6816,0.3169,0.3446,0.4060,20000,0,18.437932
4,5,1.238638,0.6805,0.3155,0.3420,0.4038,20000,0,18.423162
5,6,1.230456,0.6737,0.3116,0.3375,0.3991,20000,0,18.406424


## 11. 📊 Results: train vs tune vs test, overall and cold vs warm

All splits are measured the same way: full slates, per-impression metrics from `utils/metrics.py`.
- **Train:** impressions **used in training** (9–13 Nov), re-scored with all their displayed candidates, so train and test are comparable.
- **Tune:** 14 Nov (used to pick the epoch).
- **Test:** 15 Nov (official dev split), scored once with the selected model.

Extra tables make the results easier to interpret:
- **Random baseline on the same slates:** MRR and nDCG depend on slate length and number of clicks, which differ between segments. Comparing each segment with random scores on its own slates shows the model's real gain.
- **Segment profile:** impressions, average slate length and clicks per impression for cold, warm and mixed.
- **By history length:** the same metrics split by how many past clicks the user has (0, 1–5, 6–20, 21–50, over 50). This is the user-side cold start: users with no history get the learned "empty user" vector,, so for a given slate they all receive the same ranking.

Score files go to `config.SCORES_DIR` (`data/scores/`).

In [14]:
def full_slates_from_impressions(imp_df, impression_ids, n, seed):
    # Expand 'N1-0 N2-1 ...' slate strings of a sample of the given impressions into candidate rows.
    pool = imp_df[imp_df.impression_id.isin(impression_ids)].drop_duplicates("impression_id")
    sample = pool.sample(n=min(n, len(pool)), random_state=seed)
    rows = sample[["impression_id", "impressions"]].assign(cand=sample["impressions"].str.split()).explode("cand")
    nid_lab = rows["cand"].str.rsplit("-", n=1)
    return pd.DataFrame({"impression_id": rows["impression_id"].to_numpy(),
                         "news_id": nid_lab.str[0].to_numpy(),
                         "label": nid_lab.str[1].astype(int).to_numpy()})

def save_scores(s, name):
    path = os.path.join(SCORE_DIR, f"{name}.parquet")
    s.to_parquet(path, index=False)
    print(f"saved {name}: {len(s):,} rows, columns {list(s.columns)}")

results, scored = {}, {}

# ---- train: training impressions on full slates ----
if COLS["impressions"] in schemas["fit_imp"]:
    slates = pd.read_parquet(os.path.join(PROC, FILES["fit_imp"]), columns=[COLS["impression_id"], COLS["impressions"]])
    slates.columns = ["impression_id", "impressions"]
    tr = full_slates_from_impressions(slates, np.unique(train_imp_ids), CFG["n_train_slate_imps"], CFG["seed"])
    del slates
    tr_H, tr_idx = build_history(fit_imp, impression_ids=tr.impression_id.unique())
    s = score_candidates(model, tr, tr_H, tr_idx)
    save_scores(s, "step4_nrms_train_sample")
    results["train (training impressions)"] = evaluate(with_scores(tr, s), "score")
else:
    print("fit_impressions has no slate column: train metrics on full slates skipped.")

# ---- tune and test ----
for split, cand, H, idx in [("tune", tune_cand, tune_H, tune_idx), ("eval", eval_cand, eval_H, eval_idx)]:
    s = score_candidates(model, cand, H, idx)
    save_scores(s, f"step4_nrms_{split}")
    scored[split] = with_scores(cand, s)
    results[split] = evaluate(scored[split], "score")
    del s
del scored["tune"]                               # only the test day is broken down further
gc.collect()

overall = pd.DataFrame(results).T
overall.to_csv(os.path.join(RUN_DIR, f"{RUN}_overall.csv"))
print("\nOverall (full slates):")
display(overall)

# ---- test by cold/warm segment, with a random baseline on the same slates ----
def random_scores(df):
    return df.assign(score=np.random.default_rng(CFG["seed"]).random(len(df)))

def segment_table(test):
    rand = random_scores(test)
    m_seg = evaluate_by_segment(test, "score", "is_cold")
    r_seg = evaluate_by_segment(rand, "score", "is_cold")
    seg = label_impression_segment(test, "is_cold")
    slate = test.groupby("impression_id").agg(slate_len=("label", "size"), clicks=("label", "sum")).join(seg)
    profile = slate.groupby("segment")[["slate_len", "clicks"]].mean().round(1)
    profile.loc["all"] = slate[["slate_len", "clicks"]].mean().round(1)
    t = pd.DataFrame({
        "impressions": m_seg["n_impressions"],
        "avg_slate_len": profile["slate_len"], "avg_clicks": profile["clicks"],
        "AUC": m_seg["auc"], "AUC_random": r_seg["auc"],
        "MRR": m_seg["mrr"], "MRR_random": r_seg["mrr"],
        "nDCG@10": m_seg["ndcg@10"], "nDCG@10_random": r_seg["ndcg@10"],
    }).reindex(["all", "cold", "warm", "mixed"]).dropna(how="all")
    t["MRR_lift"] = (t["MRR"] - t["MRR_random"]).round(4)
    return t

# ---- test by user history length (user-side cold start) ----
HIST_BINS = [-1, 0, 5, 20, 50, np.inf]
HIST_LABELS = ["0 (no history)", "1-5", "6-20", "21-50", "over 50"]

def history_table(test, imp_df):
    # Full history length (before the 50-click cap) of each impression's user
    n_hist = imp_df.drop_duplicates("impression_id").set_index("impression_id")["history"].map(lambda h: len(parse_history(h)))
    rows = []
    for df, name in [(test, "model"), (random_scores(test), "random")]:
        per = evaluate(df, "score", per_impression=True)
        per["history"] = pd.cut(per["impression_id"].map(n_hist), HIST_BINS, labels=HIST_LABELS)
        rows.append(per.groupby("history", observed=False)[["auc", "mrr"]].mean().add_suffix(f"_{name}"))
        if name == "model":
            counts = per.groupby("history", observed=False).size().rename("impressions")
    t = pd.concat([counts] + rows, axis=1)
    t["MRR_lift"] = t["mrr_model"] - t["mrr_random"]
    t["share_of_impressions"] = t["impressions"] / t["impressions"].sum()
    return t[["impressions", "share_of_impressions", "auc_model", "mrr_model", "mrr_random", "MRR_lift"]].round(4)

test = scored["eval"]
by_segment = segment_table(test)
by_segment.to_csv(os.path.join(RUN_DIR, f"{RUN}_eval_by_segment.csv"))
print("\nTest (15 Nov) by impression segment (cold = every clicked article was cold):")
display(by_segment)

by_history = history_table(test, eval_imp)
by_history.to_csv(os.path.join(RUN_DIR, f"{RUN}_eval_by_history.csv"))
print("\nTest (15 Nov) by the user's number of past clicks:")
display(by_history)

# ---- how much of each day is new content? (explains the tune -> test drop) ----
freshness = pd.DataFrame({
    "cold share of candidates": [tune_cand.is_cold.mean(), eval_cand.is_cold.mean()],
    "cold share of clicks": [tune_cand.loc[tune_cand.label == 1, "is_cold"].mean(),
                             eval_cand.loc[eval_cand.label == 1, "is_cold"].mean()],
    "AUC": [results["tune"]["auc"], results["eval"]["auc"]],
}, index=["tune (14 Nov)", "test (15 Nov)"]).round(4)
display(freshness)

saved step4_nrms_train_sample: 799,703 rows, columns ['impression_id', 'news_id', 'score']
saved step4_nrms_tune: 17,400,106 rows, columns ['impression_id', 'news_id', 'score']
saved step4_nrms_eval: 14,085,557 rows, columns ['impression_id', 'news_id', 'score']

Overall (full slates):


,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped
train (training impressions),0.7639,0.3948,0.4452,0.5016,20000.0,0.0
tune,0.6835,0.3168,0.3451,0.4062,431517.0,0.0
eval,0.6482,0.3054,0.3334,0.3996,376471.0,0.0



Test (15 Nov) by impression segment (cold = every clicked article was cold):


,impressions,avg_slate_len,avg_clicks,AUC,AUC_random,MRR,MRR_random,nDCG@10,nDCG@10_random,MRR_lift
all,376471,37.4,1.5,0.6482,0.5008,0.3054,0.2189,0.3996,0.2866,0.0865
cold,77199,24.5,1.1,0.6188,0.5006,0.3900,0.2986,0.4801,0.3819,0.0914
warm,262807,37.7,1.4,0.6538,0.5010,0.2969,0.2102,0.3875,0.2734,0.0867
mixed,36465,63.1,3.3,0.6700,0.5002,0.1876,0.1128,0.3167,0.1796,0.0748



Test (15 Nov) by the user's number of past clicks:


,impressions,share_of_impressions,auc_model,mrr_model,mrr_random,MRR_lift
history,,,,,,
0 (no history),11270,0.0299,0.5544,0.2349,0.2181,0.0169
1-5,55262,0.1468,0.6050,0.3188,0.2565,0.0623
6-20,131368,0.3489,0.6453,0.3166,0.2289,0.0877
21-50,103557,0.2751,0.6684,0.3060,0.2063,0.0997
over 50,75014,0.1993,0.6711,0.2858,0.1911,0.0947


,cold share of candidates,cold share of clicks,AUC
tune (14 Nov),0.2099,0.2543,0.6835
test (15 Nov),0.1793,0.2346,0.6482


### Interpretation

- **Overall:** train 0.764 → tune 0.684 → test **0.648** AUC, with epoch 4 of 6 selected on the tune day. All three are measured on full slates, so the gap is a genuine generalisation gap, not a candidate-set artefact.
- **Tune → test drop (0.035 AUC):** 14 Nov is the day right after training, while by 15 Nov most of the articles shown first appeared after the training window (median article lifespan is about 13 hours, interim report Section 3.4). The test day even has fewer cold candidates (17.9% vs 21.0%), so the drop comes from articles the model has not seen in training, not from the cold share.
- **Cold vs warm:** cold-click impressions reach AUC 0.619 against 0.654 for warm ones, both far above random (0.50). The gap, 0.035, is about half that of title-only NRMS (0.064, section 11a). Segments are compared by AUC or lift over random, since raw MRR and nDCG also depend on slate length and click count.
- **By history length:** AUC rises from 0.554 for users with no history (3.0% of impressions, who all get the same learned vector) to 0.605 with 1–5 past clicks and about 0.67 with more than 20. Personalisation needs a few clicks; content alone does little for users the model knows nothing about.

## 11a. Comparison with standard NRMS (title only)

Standard title-only NRMS (`BASELINE_RUN`) is trained by the same `train_nrms` call as the main model, with `use_category=False, use_entities=False`: the same 1.5M clicks, negatives, epochs and epoch selection, so the only difference is the article inputs. With `RESUME = True` the finished run is reloaded from its checkpoint; on a fresh run it is trained here. Both models are scored on the same tune and test impressions.

The architecture is justified on the **tune day**: the comparison on 14 Nov is printed first. Test results are then shown for both models on equal terms.

In [18]:
# Standard title-only NRMS: same data, negatives, epochs and epoch selection, with the two added inputs switched off
base_cfg = dict(CFG, use_category=False, use_entities=False)
baseline, baseline_log = train_nrms(base_cfg, BASELINE_RUN, fit_H, train_H_rows, train_C)

base_results, base_scored = {}, {}
for split, cand, H, idx in [("tune", tune_cand, tune_H, tune_idx), ("eval", eval_cand, eval_H, eval_idx)]:
    base_scored[split] = with_scores(cand, score_candidates(baseline, cand, H, idx))
    base_results[split] = evaluate(base_scored[split], "score")
print(f"Tune AUC (14 Nov): standard NRMS {base_results['tune']['auc']:.4f} | "
      f"our model {results['tune']['auc']:.4f} | difference {results['tune']['auc'] - base_results['tune']['auc']:+.4f}")

vs_baseline = pd.concat({"Standard NRMS (title only)": pd.DataFrame(base_results).T,
                         "Our model (title + category + entities)": pd.DataFrame({k: results[k] for k in ("tune", "eval")}).T})
vs_baseline.to_csv(os.path.join(RUN_DIR, f"{RUN}_vs_{BASELINE_RUN}.csv"))
display(vs_baseline.round(4))

base_segment = segment_table(base_scored["eval"])
seg_compare = pd.DataFrame({"impressions": by_segment["impressions"],
                            "AUC standard NRMS": base_segment["AUC"], "AUC our model": by_segment["AUC"],
                            "difference": by_segment["AUC"] - base_segment["AUC"]}).round(4)
seg_compare.to_csv(os.path.join(RUN_DIR, f"{RUN}_vs_{BASELINE_RUN}_by_segment.csv"))
print("\nTest (15 Nov) AUC by segment:")
display(seg_compare)

del baseline, base_scored, base_segment
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Resumed nrms_main_v2 after epoch 6
nrms_main_v2: using epoch 6 (tune AUC 0.6750)
Tune AUC (14 Nov): standard NRMS 0.6757 | our model 0.6835 | difference +0.0078


auc     mrr  ndcg@5  ndcg@10  \
Standard NRMS (title only)              tune  0.6757  0.3109  0.3365   0.3984   
                                        eval  0.6288  0.2907  0.3162   0.3829   
Our model (title + category + entities) tune  0.6835  0.3168  0.3451   0.4062   
                                        eval  0.6482  0.3054  0.3334   0.3996   

                                              n_impressions  n_skipped  
Standard NRMS (title only)              tune       431517.0        0.0  
                                        eval       376471.0        0.0  
Our model (title + category + entities) tune       431517.0        0.0  
                                        eval       376471.0        0.0


Test (15 Nov) AUC by segment:


,impressions,AUC standard NRMS,AUC our model,difference
all,376471,0.6288,0.6482,0.0194
cold,77199,0.5767,0.6188,0.0421
warm,262807,0.6411,0.6538,0.0127
mixed,36465,0.6503,0.6700,0.0197


**Interpretation:** on the tune day, the added inputs raise AUC from 0.6757 to 0.6835 (+0.008), so the architecture is kept on the tune day's evidence before the test day is looked at. On the test day the gain is larger, 0.629 → 0.648 (+0.019), and largest on cold-click impressions: 0.577 → 0.619 (+0.042), against +0.013 on warm ones. The cold–warm gap almost halves (0.064 → 0.035): content that exists at publication time helps most where click history is missing, and the gain grows on a day with more articles the model has never seen.

## 11b. Export article vectors (shared)

The final model's 256-d article vectors (the news encoder output, i.e. what the model compares with the user vector) are saved for the rest of the team:
- `data/embeddings/step4_nrms_news_vectors.npy`: float32 array `[n_articles, 256]`
- `data/embeddings/step4_nrms_news_ids.parquet`: `news_id` in the same row order
- `data/embeddings/step4_nrms_news_vectors.json`: what the vectors are and how they were made

The vectors are **not normalised**. For cosine similarity, divide each row by its length first.

In [19]:
FINAL_MODEL, FINAL_NAME, FINAL_TRAINED_ON = model, RUN, "fit 9-13 Nov"
vecs = encode_all_news(FINAL_MODEL).cpu().numpy()[1:].astype(np.float32)   # drop padding row 0
ids = news[["news_id"]].copy()
assert len(vecs) == len(ids) and ids.news_id.is_unique

np.save(os.path.join(EMB_DIR, "step4_nrms_news_vectors.npy"), vecs)
ids.to_parquet(os.path.join(EMB_DIR, "step4_nrms_news_ids.parquet"), index=False)
with open(os.path.join(EMB_DIR, "step4_nrms_news_vectors.json"), "w") as f:
    json.dump({"model": FINAL_NAME, "epoch": FINAL_MODEL.best_epoch, "trained_on": FINAL_TRAINED_ON,
               "dim": int(vecs.shape[1]), "n_articles": int(len(vecs)), "normalised": False,
               "source": "article vector of the news encoder (title words, word2vec-initialised and fine-tuned; "
                         "category and category_subcategory embeddings; confidence-weighted TransE entity vector)",
               "row_order": "same as step4_nrms_news_ids.parquet"}, f, indent=2)

norms = np.linalg.norm(vecs, axis=1)
print(f"Saved {vecs.shape[0]:,} article vectors of size {vecs.shape[1]} to {EMB_DIR}")
print(f"Vector length: median {np.median(norms):.2f}, min {norms.min():.2f}, max {norms.max():.2f}")

Saved 104,151 article vectors of size 256 to /content/drive/MyDrive/BT4222_Project/data/embeddings
Vector length: median 4.62, min 1.67, max 12.69


## 12. Ablations

Each variant changes one setting of the full model, so any difference can be attributed to that setting:
- **− entities:** no entity encoder.
- **Entities averaged:** the entity encoder's attention layers replaced by a confidence-weighted average, to test whether attention over entities is worth it.
- **− category:** no category embeddings.
- **Title only:** both removed, i.e. standard NRMS.

**Table B** (second cell) tests the title- and user-encoder settings (history length, number of heads, abstract words). They were trained on the title-only model, and those parts of the architecture are unchanged in the full model; the saved runs are reloaded and rescored here with the same paired standard errors, so no retraining is needed.

To fit the time budget, ablations train on 200k clicks for 2 epochs and are compared on the **tune day only**; the test day is not used to choose between variants. All variants use the same seed, the same training sample and the same tune impressions. Finished ablations reload from their checkpoints.

Besides overall AUC, the table reports AUC on **cold-click impressions** (every clicked article was cold), since that is where the content inputs should matter most.

**Is a difference real?** Because every variant is scored on the same tune impressions, the difference from the base is computed impression by impression, and its standard error (SE) is the standard deviation of those differences ÷ √(number of impressions). A difference smaller than about 2 SE cannot be told apart from noise.

In [20]:
ABL_BASE = dict(CFG, n_train_clicks=200_000, epochs=2)
ABLATIONS = {
    "full model (title + category + entities)": {},
    # inputs, removed one at a time
    "- entities":                               {"use_entities": False},
    "entities averaged, no entity encoder":     {"entity_pool": "mean"},
    "- category":                               {"use_category": False},
    "title only (standard NRMS)":               {"use_category": False, "use_entities": False,
                                                 "run": "abl_base_title_16_heads_hist_50"},   # reuses the earlier title-only ablation (same data and settings)
}

def tune_per_impression(m, t_H, t_idx):
    # per-impression metrics on the fixed tune subset, with each impression's cold/warm segment
    df = with_scores(tune_small, score_candidates(m, tune_small, t_H, t_idx))
    per = evaluate(df, "score", per_impression=True).set_index("impression_id")
    per["segment"] = label_impression_segment(df, "is_cold").reindex(per.index)
    return per

def compare_to_base(per, base):
    # mean metric per variant + paired difference from the base on the same impressions, with its SE
    def paired(a, b):
        d = (a - b).dropna()
        return d.mean(), d.std(ddof=1) / np.sqrt(len(d))
    row = {"auc": per.auc.mean(), "mrr": per.mrr.mean(), "ndcg@10": per["ndcg@10"].mean(),
           "auc_cold": per.loc[per.segment == "cold", "auc"].mean(),
           "auc_warm": per.loc[per.segment == "warm", "auc"].mean()}
    row["AUC_vs_base"], row["SE"] = paired(per.auc, base.auc)
    cold = per.segment == "cold"
    row["cold_AUC_vs_base"], row["cold_SE"] = paired(per.auc[cold], base.auc[cold])
    return row

def run_ablation(name, overrides):
    global TOKENS, CFG
    overrides = dict(overrides)
    run = overrides.pop("run", None) or "abl3_" + re.sub(r"[^a-z0-9]+", "_", name.lower().replace("- ", "no ").replace("+ ", "plus ")).strip("_")
    cfg = dict(ABL_BASE, **overrides)
    saved = (TOKENS, CFG)
    try:
        CFG = cfg                         # build_history and the model read the history length from CFG
        if cfg["text_fields"] != saved[1]["text_fields"]:
            TOKENS = build_tokens(cfg["text_fields"], cfg["field_lens"], VOCAB)
        imp_ids, C = build_train_samples(fit_cand, cfg["n_train_clicks"], cfg["neg_k"], cfg["seed"])
        H, idx = build_history(fit_imp, impression_ids=np.unique(imp_ids))
        t_H, t_idx = build_history(tune_imp)
        m, log = train_nrms(cfg, run, H, idx.reindex(imp_ids).to_numpy(), C,
                            tune=(tune_small, t_H, t_idx), log_every=0)
        per = tune_per_impression(m, t_H, t_idx)
        best_epoch = m.best_epoch
        del m
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return best_epoch, per
    finally:
        TOKENS, CFG = saved

if RUN_ABLATIONS:
    abl_runs = {}
    for name, o in ABLATIONS.items():
        if dict(ABL_BASE, **o)["use_entities"] and ENT_INPUTS is None:
            print(f"skipped '{name}': no entity_embedding.vec found (section 3b)")
            continue
        abl_runs[name] = run_ablation(name, o)
    base_per = next(iter(abl_runs.values()))[1]                  # first row = full model
    ablation_table = pd.DataFrame({name: {"best_epoch": ep, **compare_to_base(per, base_per)}
                                   for name, (ep, per) in abl_runs.items()}).T
    ablation_table["n_cold_impressions"] = int((base_per.segment == "cold").sum())
    ablation_table = ablation_table.astype(float).round(4)
    ablation_table.to_csv(os.path.join(RUN_DIR, "ablation_inputs_tune.csv"))
    print("Inputs (base = full model):")
    display(ablation_table)

Resumed abl3_full_model_title_plus_category_plus_entities after epoch 2
abl3_full_model_title_plus_category_plus_entities: using epoch 2 (tune AUC 0.6578)
Resumed abl3_no_entities after epoch 2
abl3_no_entities: using epoch 2 (tune AUC 0.6568)
Resumed abl3_entities_averaged_no_entity_encoder after epoch 2
abl3_entities_averaged_no_entity_encoder: using epoch 2 (tune AUC 0.6518)
Resumed abl3_no_category after epoch 2
abl3_no_category: using epoch 2 (tune AUC 0.6406)
Resumed abl_base_title_16_heads_hist_50 after epoch 2
abl_base_title_16_heads_hist_50: using epoch 2 (tune AUC 0.6238)
Inputs (base = full model):


,best_epoch,auc,mrr,ndcg@10,auc_cold,auc_warm,AUC_vs_base,SE,cold_AUC_vs_base,cold_SE,n_cold_impressions
full model (title + category + entities),2.0,0.6578,0.2954,0.3801,0.6179,0.6737,0.0000,0.0000,0.0000,0.0000,4590.0
- entities,2.0,0.6568,0.2931,0.3788,0.6365,0.6640,-0.0011,0.0011,0.0186,0.0031,4590.0
"entities averaged, no entity encoder",2.0,0.6518,0.2914,0.3758,0.6147,0.6659,-0.0060,0.0011,-0.0032,0.0029,4590.0
- category,2.0,0.6406,0.2770,0.3589,0.6045,0.6539,-0.0172,0.0015,-0.0133,0.0041,4590.0
title only (standard NRMS),2.0,0.6238,0.2646,0.3445,0.5747,0.6430,-0.0340,0.0016,-0.0432,0.0045,4590.0


### Table B: title- and user-encoder settings (base = title-only NRMS)

The four title-only runs (same 200k clicks, 2 epochs, seed and tune impressions as above) are reloaded from their checkpoints and rescored, so each difference from the title-only base gets a paired standard error. A run whose checkpoint files are missing is skipped rather than retrained.

In [21]:
TITLE_ONLY = {"use_category": False, "use_entities": False}
ARCH_ABLATIONS = {
    "title only, 16 heads, history 50 (base)": {**TITLE_ONLY, "run": "abl_base_title_16_heads_hist_50"},
    "history 10":       {**TITLE_ONLY, "max_hist": 10, "run": "abl_history_10"},
    "4 heads":          {**TITLE_ONLY, "n_heads": 4, "run": "abl_4_heads"},
    "title + abstract": {**TITLE_ONLY, "text_fields": ("title", "abstract"), "field_lens": (30, 50),
                         "run": "abl_title_abstract"},
}

arch_runs = {}
for name, o in ARCH_ABLATIONS.items():
    files = [os.path.join(RUN_DIR, f"{o['run']}_{k}.pt") for k in ("best", "last")]
    if not all(os.path.exists(f) for f in files):
        print(f"skipped '{name}': checkpoint files for {o['run']} not found (not retrained here)")
        continue
    arch_runs[name] = run_ablation(name, o)

if arch_runs:
    arch_base = next(iter(arch_runs.values()))[1]          # first row = title-only base
    arch_table = pd.DataFrame({n: {"best_epoch": ep, **compare_to_base(per, arch_base)}
                               for n, (ep, per) in arch_runs.items()}).T.astype(float).round(4)
    arch_table.to_csv(os.path.join(RUN_DIR, "ablation_encoder_settings_tune.csv"))
    print("Title- and user-encoder settings (base = title-only NRMS):")
    display(arch_table)

Resumed abl_base_title_16_heads_hist_50 after epoch 2
abl_base_title_16_heads_hist_50: using epoch 2 (tune AUC 0.6238)
Resumed abl_history_10 after epoch 2
abl_history_10: using epoch 2 (tune AUC 0.6202)
Resumed abl_4_heads after epoch 2
abl_4_heads: using epoch 2 (tune AUC 0.6286)
Resumed abl_title_abstract after epoch 2
abl_title_abstract: using epoch 2 (tune AUC 0.6221)
Title- and user-encoder settings (base = title-only NRMS):


,best_epoch,auc,mrr,ndcg@10,auc_cold,auc_warm,AUC_vs_base,SE,cold_AUC_vs_base,cold_SE
"title only, 16 heads, history 50 (base)",2.0,0.6238,0.2646,0.3445,0.5747,0.6430,0.0000,0.0000,0.0000,0.0000
history 10,2.0,0.6202,0.2671,0.3474,0.5661,0.6411,-0.0036,0.0010,-0.0085,0.0030
4 heads,2.0,0.6286,0.2706,0.3516,0.5753,0.6492,0.0048,0.0009,0.0007,0.0027
title + abstract,2.0,0.6221,0.2658,0.3461,0.5694,0.6420,-0.0017,0.0014,-0.0052,0.0040


**Interpretation (Table A, inputs; 200k clicks, 2 epochs):**
- **Category gives most of the gain.** Removing it costs 0.017 AUC overall (about 11 SE) and 0.013 on cold-click impressions (about 3 SE). Removing both added inputs costs 0.034 overall and 0.043 on cold clicks.
- **Entities help warm articles, not cold ones, at this budget.** Removing them changes overall AUC by only −0.001 (1 SE, not distinguishable from zero), but raises cold-click AUC by 0.019 (about 6 SE) and lowers warm AUC by 0.010. A possible reason: entity vectors let the model learn which people and stories were popular during the training days, which helps on ongoing stories and less on new ones.
- **Attention over entities beats a plain average:** replacing the entity encoder with a confidence-weighted average costs 0.006 overall (about 5 SE); on cold clicks the difference is within noise.
- **Caveat:** one seed, a reduced budget and 4,590 cold-click tune impressions, so these rank design choices rather than estimate full-scale gains. The full-scale comparison (section 11a) tests both inputs together: +0.042 test AUC on cold clicks.

**Table B (encoder settings):** every variant moves tune AUC by less than 0.005 from the title-only base (see the SE column), while training on 1.5M instead of 200k clicks raises it by 0.05 (0.676 vs 0.624). At this budget data matters far more than these settings, so the standard NRMS values are kept.

## 13. Inside the model: attention weights

For a few test impressions:
- **Word attention:** which title words the news encoder weighted most for the clicked article.
- **History attention:** which past clicks the user encoder weighted most.
- **Entity attention:** which of the clicked article's entities the entity encoder weighted most.

To read a weight, compare it with equal attention: 1 ÷ the number of items (about 0.12 for an 8-word title, 0.02 for 50 history articles).

Caveat: attention weights show what the model relied on to make a prediction. They are not verified reasons why the user clicked.

In [22]:
# Free the large training tables: from here on only the model, news and the test split are needed.
for _name in ["fit_cand", "fit_imp", "fit_H", "train_C", "train_H_rows", "train_imp_ids", "tr", "tr_H", "data",
              ]:
    globals().pop(_name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

INV_VOCAB = {i: w for w, i in VOCAB.items()}
NEWS_VECS = encode_all_news(model)          # every article encoded once, reused below

@torch.no_grad()
def explain(model, impression_id, top=5):
    model.eval()
    hist = torch.as_tensor(eval_H[[eval_idx[impression_id]]], device=DEVICE)
    _, hw = model.user_encoder(NEWS_VECS[hist], hist > 0, return_weights=True)
    hrows, hw = hist[0].cpu().numpy(), hw[0].float().cpu().numpy()
    n_hist = int((hrows > 0).sum())
    print(f"Impression {impression_id}: most-attended history articles (equal attention = {1 / max(n_hist, 1):.2f})")
    for j in np.argsort(-hw)[:3]:
        if hrows[j] > 0:
            print(f"   {hw[j]:.2f}  {news.loc[hrows[j] - 1, 'title']}")
    clicked = eval_cand.loc[(eval_cand.impression_id == impression_id) & (eval_cand.label == 1), "news_id"]
    for nid in clicked:
        row = int(NEWS_ROW[nid])
        tok = model.tokens[[row]]
        _, ww = model.news_encoder(tok, return_weights=True)
        words = [(INV_VOCAB[int(t)], float(w)) for t, w in zip(tok[0].cpu(), ww[0].float().cpu()) if int(t) > 0]
        print(f"Clicked: {news.loc[row - 1, 'title']}")
        print("   top words:", ", ".join(f"{w} ({a:.2f})" for w, a in sorted(words, key=lambda x: -x[1])[:top]))
        if getattr(model, "use_ent", False) and HAS_ENT[row] > 0 and model.entity_encoder.pool == "attention":
            _, ew = model.encode_entities(torch.as_tensor([row], device=DEVICE), return_weights=True)
            slots = ENT_INPUTS["ent_idx"][row]
            ents = [(ENT_LABEL.get(ENT_INPUTS["qids"][i - 1], "?"), float(w))
                    for i, w in zip(slots, ew[0].float().cpu()) if i > 0]
            print("   entities:", ", ".join(f"{e} ({a:.2f})" for e, a in sorted(ents, key=lambda x: -x[1])))

examples = eval_cand.loc[eval_cand.label == 1, "impression_id"].drop_duplicates().sample(3, random_state=CFG["seed"])
for imp in examples:
    explain(model, imp)
    print()

Impression 90453: most-attended history articles (equal attention = 0.07)
   0.07  Former NBA first-round pick Jim Farmer arrested in sex sting operation
   0.07  The richest country music stars
   0.07  Charlize Theron Says Playing Megyn Kelly Was 'Harder' Than Portraying a Serial Killer
Clicked: The Unlikely Star of My Family's Thanksgiving Table
   top words: thanksgiving (0.13), table (0.13), the (0.13), star (0.12), of (0.12)
   entities: Nigella Lawson (0.35), Southern United States (0.33), Stuffing (0.31)

Impression 210745: most-attended history articles (equal attention = 0.02)
   0.02  Stocks close higher on optimism over China trade deal
   0.02  Stocks close higher on optimism over China trade deal
   0.02  Here Are the Biggest Deals We're Anticipating for Black Friday
Clicked: This Easy Trick Will Defrost Your Car Window In Seconds
   top words: defrost (0.11), seconds (0.10), your (0.10), trick (0.10), this (0.10)

Impression 192076: most-attended history articles (equal 

### 13b. Attention in numbers

Measured over thousands of test impressions:
1. **History attention vs interests:** how often the *most-attended* history article has the same category as the clicked article, compared with a *random* history article.
2. **Word attention:** average weight on stopwords (*the, my, of…*) vs content words, relative to equal attention (1.0 = every word weighted the same).

In [23]:
STOPWORDS = set("the a an of to in on for and or with at by from is are was were be been this that these those it its "
                "he she they his her their my your our you we i as after says how why what when who new".split())
CATEGORY = news["category"].to_numpy()          # article row r -> CATEGORY[r - 1]

@torch.no_grad()
def attention_stats(model, n_imps=3000, seed=CFG["seed"]):
    model.eval()
    rng = np.random.default_rng(seed)
    clicks = eval_cand.loc[eval_cand.label == 1, ["impression_id", "news_id"]]
    imps = clicks["impression_id"].drop_duplicates()
    imps = imps.sample(n=min(n_imps, len(imps)), random_state=seed).to_numpy()
    clicked_cats = (clicks.assign(cat=CATEGORY[NEWS_ROW.reindex(clicks.news_id).to_numpy() - 1])
                          .groupby("impression_id")["cat"].agg(set))

    # 1. history attention: top-attended vs random history article
    top_hit, rand_hit = [], []
    for s in range(0, len(imps), 1000):
        batch = imps[s:s + 1000]
        hist = torch.as_tensor(eval_H[eval_idx.reindex(batch).to_numpy()], device=DEVICE)
        _, hw = model.user_encoder(NEWS_VECS[hist], hist > 0, return_weights=True)
        hist, hw = hist.cpu().numpy(), hw.float().cpu().numpy()
        for imp, h, w in zip(batch, hist, hw):
            real = np.flatnonzero(h > 0)
            if len(real) < 2:
                continue
            cats = clicked_cats[imp]
            top_hit.append(CATEGORY[h[real[np.argmax(w[real])]] - 1] in cats)
            rand_hit.append(CATEGORY[h[rng.choice(real)] - 1] in cats)

    # 2. word attention: stopwords vs content words (weight x number of words; 1.0 = equal)
    rows = NEWS_ROW.reindex(clicks.loc[clicks.impression_id.isin(imps), "news_id"].unique()).to_numpy()
    stop_w, content_w = [], []
    for s in range(0, len(rows), 2000):
        tok = model.tokens[torch.as_tensor(rows[s:s + 2000], device=DEVICE)]
        _, ww = model.news_encoder(tok, return_weights=True)
        tok, ww = tok.cpu().numpy(), ww.float().cpu().numpy()
        for t, w in zip(tok, ww):
            real = t > 0
            n = real.sum()
            for ti, wi in zip(t[real], w[real]):
                (stop_w if INV_VOCAB[int(ti)] in STOPWORDS else content_w).append(wi * n)

    return pd.DataFrame({
        "value": [np.mean(top_hit), np.mean(rand_hit), np.mean(stop_w), np.mean(content_w)],
        "n": [len(top_hit), len(rand_hit), len(stop_w), len(content_w)],
    }, index=["history: top-attended article shares clicked category",
              "history: random article shares clicked category",
              "words: relative weight on stopwords (1 = equal)",
              "words: relative weight on content words (1 = equal)"]).round(3)

attention_table = attention_stats(model)
attention_table.to_csv(os.path.join(RUN_DIR, f"{RUN}_attention_stats.csv"))
display(attention_table)

,value,n
history: top-attended article shares clicked category,0.217,2861
history: random article shares clicked category,0.257,2861
words: relative weight on stopwords (1 = equal),0.978,1669
words: relative weight on content words (1 = equal),1.007,5394


**Interpretation:** history attention does not pick out the clicked topic: the most-attended history article shares the clicked article's category 21.7% of the time, slightly below a random history article (25.7%; title-only NRMS: 28.3% vs 25.7%). NRMS builds one user vector for all candidates, so its history attention cannot depend on the article being scored; with the category now inside every article vector, the attention appears to weight other signals than topic. Word attention stays close to uniform (stopwords 0.98 vs content words 1.01 relative weight), and the example entity weights are close to even. The weights show what the model used to predict, not verified reasons for a click; candidate-aware user attention is a natural extension.

## 14. Summary and hand-off

| | Value |
|---|---|
| Model | NRMS news and user encoders; article vector from title words + category/subcategory + entity encoder (TransE + entity-linking features) |
| Training data | 1.5M clicks from fit (9–13 Nov), 1 click + 4 same-impression negatives |
| Selected epoch | 4 of 6 (best tune AUC on the 20k-impression tune subset) |
| Test (15 Nov), full slates | AUC 0.648, MRR 0.305, nDCG@5 0.333, nDCG@10 0.400 |
| Train / tune / test AUC | 0.764 / 0.684 / 0.648 |
| Cold vs warm (test AUC) | 0.619 vs 0.654 (title-only NRMS: 0.577 vs 0.641) |
| vs title-only NRMS | tune +0.008, test +0.019, cold-click test +0.042 |
| Ablations (200k clicks) | category gives most of the gain; entities help warm but not cold articles; entity attention beats averaging |

**Files for the team**
- `data/scores/step4_nrms_eval.parquet`: test-day scores, 14,085,557 rows, `impression_id, news_id, score` (raw dot product, higher = more relevant, can be negative). Rescale within each impression (e.g. min-max) before combining with similarities in MMR. A sigmoid turns a score into a value between 0 and 1 without changing the ranking. Because training compares each click with 4 sampled negatives, that value ranks well but is not a calibrated click probability.
- `data/scores/step4_nrms_tune.parquet`, `step4_nrms_train_sample.parquet`: tune-day and training-impression scores.
- `data/embeddings/step4_nrms_news_vectors.npy` + `step4_nrms_news_ids.parquet`: the model's article vectors. For a fair diversity measure, MMR should optimise with one embedding and diversity should be evaluated with a different, independent one (e.g. step 3's text embeddings or TF-IDF), plus category counts.